# 05 - Collaborative Filtering (LightFM)

Trains `SpotifyHybridCollaborative` (`src/models/collaborative.py`) - a thin
wrapper around `LightFM` with WARP loss - on the interaction matrix, with a
held-out test split so we can measure whether it actually learns anything
beyond memorizing the training playlists.

In [1]:
import sys
sys.path.insert(0, '..')
import numpy as np
from scipy import sparse
from lightfm.cross_validation import random_train_test_split
from lightfm.evaluation import precision_at_k, auc_score
from src.models.collaborative import SpotifyHybridCollaborative

interactions = sparse.load_npz('../data/processed/interaction_matrix.npz').tocoo()
print("Full interaction matrix:", interactions.shape, "nnz:", interactions.nnz)

train, test = random_train_test_split(interactions, test_percentage=0.2, random_state=np.random.RandomState(42))
print("Train nnz:", train.nnz, " Test nnz:", test.nnz)

Full interaction matrix: (674, 16804) nnz: 225776
Train nnz: 180620  Test nnz: 45156


In [2]:
model = SpotifyHybridCollaborative(no_components=32, loss='warp', lr=0.05)
model.fit(train, epochs=15)

Epoch:   0%|          | 0/15 [00:00<?, ?it/s]

Epoch:   7%|▋         | 1/15 [00:00<00:09,  1.44it/s]

Epoch:  13%|█▎        | 2/15 [00:01<00:09,  1.43it/s]

Epoch:  20%|██        | 3/15 [00:02<00:08,  1.42it/s]

Epoch:  27%|██▋       | 4/15 [00:02<00:07,  1.41it/s]

...

Epoch:  80%|████████  | 12/15 [00:09<00:02,  1.16it/s]

Epoch:  87%|████████▋ | 13/15 [00:10<00:01,  1.17it/s]

Epoch:  93%|█████████▎| 14/15 [00:11<00:00,  1.16it/s]

Epoch: 100%|██████████| 15/15 [00:12<00:00,  1.13it/s]

Epoch: 100%|██████████| 15/15 [00:12<00:00,  1.23it/s]

## Evaluation

`precision_at_k` = fraction of the top-K recommended tracks per playlist that are actually held-out tracks from that playlist. `AUC` = probability a random held-out track ranks above a random track the playlist never had.

In [3]:
# lightfm's evaluation metrics score every item in the catalog for every
# user, so over an 80k+ track catalog and 3.3k playlists this takes a few
# minutes on a single CPU core - that's expected, not a bug.
train_precision = precision_at_k(model.model, train, k=10).mean()
test_precision = precision_at_k(model.model, test, k=10, train_interactions=train).mean()

train_auc = auc_score(model.model, train).mean()
test_auc = auc_score(model.model, test, train_interactions=train).mean()

print(f"Precision@10  train: {train_precision:.4f}   test: {test_precision:.4f}")
print(f"AUC           train: {train_auc:.4f}   test: {test_auc:.4f}")

Precision@10  train: 0.3037   test: 0.0040
AUC           train: 0.9161   test: 0.5353


**Reading these numbers honestly:** training performance is very strong,
while test performance is much weaker - that gap is real and worth
understanding rather than glossing over. Test AUC above 0.5 means the model
is ranking held-out tracks better than chance (it has learned *some* real
structure), but precision@10 on the test set is close to zero, because most
playlists only have a handful of true held-out tracks against a catalog of
16k+ - getting several of the top 10 exactly right is a very high bar at
this scale.

The bigger factor is the playlist-construction method from notebook 02:
these are genre-chunked, popularity-sorted playlists with added crossover
tracks, not real listening behavior. Once 20% of edges are held out for
testing, there's only so much genuine collaborative signal left to
generalize from. This is a concrete, honest illustration of why this
project matters more with real interaction data: swap
`data/processed/playlist_track_interactions.csv` for real MPD playlist
logs (same `pid` / `track_uri` columns) and this same evaluation cell will
tell you how the model performs on real human behavior instead of a
genre-based proxy.

## Save the trained model

In [4]:
import pickle
with open('../data/processed/lightfm_model.pkl', 'wb') as f:
    pickle.dump(model, f)
print("Saved data/processed/lightfm_model.pkl")

Saved data/processed/lightfm_model.pkl
